# 04 · Reconstruction analysis: experiments E1–E5

**Goal:** read the experiment results (`scripts/run_experiments.py`) and answer the research questions RQ1–RQ4 plus the bundle-adjustment gain. The written conclusions are in `docs/experiments.md`; this notebook shows the evidence and regenerates the comparison plots from the CSVs.

Set `EXPERIMENTS_DIR` (or `MV3D_EXPERIMENTS`) to a run directory; by default the newest `outputs/experiments/*` is used. Run the experiments first:

```bash
python scripts/run_experiments.py
```

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

from src.evaluation.reports import LABELS
from src.run import read_json

runs = sorted(Path('../outputs/experiments').glob('*_experiments'))
EXPERIMENTS_DIR = Path(os.environ.get('MV3D_EXPERIMENTS', runs[-1] if runs else ''))
assert (EXPERIMENTS_DIR / 'config.yaml').is_file(), 'run scripts/run_experiments.py first'

experiments = {}
for meta_path in sorted(EXPERIMENTS_DIR.glob('*/experiment.json')):
    meta = read_json(meta_path)
    experiments[meta['name']] = (meta, pd.read_csv(meta_path.parent / 'experiment_results.csv'))
print(EXPERIMENTS_DIR)
print({name: len(table) for name, (_, table) in experiments.items()})

## How runs are scored

Pose errors compare each registered camera with ground truth after the best similarity alignment. *Completeness* is the share of reference points (every verified SIFT track over all images, triangulated with the **ground-truth cameras**) that have a reconstructed point within 1 % of the reference's extent; *accuracy* is the median distance from our points to the reference. The reference shares our features and matches, so it is an internal reference, not an independent surface scan.

In [ ]:
def show(name, columns):
    meta, table = experiments[name]
    display(Markdown(f"**{meta['name']}** — {meta['question']}"))
    shown = ['setting'] + [c for c in columns if c in table]
    display(table[shown].rename(columns=LABELS).round(3))
    display(Image(str(EXPERIMENTS_DIR / name / 'metrics.png')))

## E1 · Image count (RQ1)

Subsets evenly spaced in file-name order (templeRing's file order is not angular order, so coverage of the ring is uneven; see docs/experiments.md). Expect a threshold rather than a smooth curve: views too far apart do not match, so they never register.

In [ ]:
show('e1_image_count', ['registered', 'filtered_points', 'mean_track_length', 'completeness',
                        'rotation_error_median_deg', 'time_total_s'])

## E2 · Matching thresholds (RQ2)

Lowe ratio × mutual check. More permissive matching verifies more pairs, but does the final cloud improve?

In [ ]:
show('e2_matching', ['verified_pairs', 'inlier_ratio_mean', 'filtered_points', 'mean_track_length',
                     'completeness', 'reprojection_mean_px'])

## E3 · Calibration accuracy (RQ3)

The bar chart is dominated by the guessed intrinsics (a −49 % focal length). The second plot shows the controlled focal-length errors on a numeric axis, with the reprojection error alongside: it hardly moves while the geometry degrades.

In [ ]:
show('e3_calibration', ['registered', 'rotation_error_median_deg', 'centre_error_median',
                        'reprojection_mean_px', 'completeness', 'accuracy_median'])

In [ ]:
_, e3 = experiments['e3_calibration']
focal = e3[e3['setting'].str.startswith('focal') | (e3['setting'] == 'calibrated')].copy()
focal['focal error (%)'] = 100 * (focal['camera.focal_scale'].fillna(1.0) - 1)
focal = focal.sort_values('focal error (%)')

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), layout='constrained')
for ax, metric in zip(axes, ['completeness', 'rotation_error_median_deg', 'reprojection_mean_px']):
    ax.plot(focal['focal error (%)'], focal[metric], marker='o')
    ax.set_xlabel('focal length error (%)')
    ax.set_ylabel(LABELS.get(metric, metric))
    ax.grid(alpha=0.3)
fig.suptitle('E3: controlled focal-length errors (guessed intrinsics omitted)')
plt.show()

## E4 · SIFT vs ORB (RQ4)

Same cap (8000), resolution, pairs and RANSAC settings. Compare by completeness and pose error: *accuracy* favours SIFT because the reference is built from SIFT tracks.

In [ ]:
show('e4_detector', ['features_mean', 'inlier_ratio_mean', 'filtered_points', 'mean_track_length',
                     'completeness', 'rotation_error_median_deg', 'reprojection_mean_px',
                     'time_total_s'])
display(Image(str(EXPERIMENTS_DIR / 'e4_detector' / 'runtime.png')))

## E5 · Bundle adjustment

Bundle adjustment on vs off (off also skips the structure refinement that follows each adjustment).

In [ ]:
show('e5_bundle_adjustment', ['rotation_error_median_deg', 'rotation_error_max_deg',
                              'centre_error_median', 'reprojection_mean_px', 'completeness',
                              'time_total_s'])
display(Image(str(EXPERIMENTS_DIR / 'e5_bundle_adjustment' / 'runtime.png')))

## Metric scale from one measured distance

The scale from a single known distance (the true distance between the first two registered cameras), relative to the scale fitted over all cameras.

In [ ]:
rows = [table[['setting', 'metric_scale_error']].assign(experiment=name)
        for name, (_, table) in experiments.items()]
scale = pd.concat(rows).dropna()
scale['metric scale error (%)'] = (100 * scale.pop('metric_scale_error')).round(2)
scale.set_index(['experiment', 'setting'])

## Summary

- **RQ1** — completeness is set by view spacing: with a median gap above ~20° between views SIFT matches fail and images do not register; with all images (median gap 8°) 47/47 register with 0.92 completeness.
- **RQ2** — a ratio of 0.7–0.8 with the mutual check is best; looser matching verifies more pairs but yields fewer, worse points.
- **RQ3** — a 5 % focal-length error halves completeness while the reprojection error stays flat; guessed intrinsics give a distorted model. Calibrate, or refine the focal length in bundle adjustment.
- **RQ4** — ORB gives 3× the points but 4× the pose error, lower completeness and 4× the runtime; SIFT is the better default here.
- **E5** — bundle adjustment cuts pose errors 4× and raises completeness from 0.38 to 0.92, at ~90 % of the runtime.
- **RQ5** (sparse vs dense, E6 with COLMAP) is still open.

Details and caveats: `docs/experiments.md`.